# ระดับ 2 — ประเมิน CFB (FFDBackbone) บน Celeb-DF v2 test set เทียบกับ paper

**Paper:** Guo et al., *Revisiting Face Forgery Detection: From Facial Representation to Forgery Detection*, IEEE TPAMI 2026
**Code:** https://github.com/zhenglab/FFDBackbone

**เป้าหมาย:** ทำซ้ำตัวเลข **Celeb-DF video-level AUC = 90.46** (Table ผลหลักของ paper) ด้วย checkpoint ของผู้เขียน
ซึ่งเทรนบน FF++ c23 เท่านั้น → Celeb-DF เป็น **unseen dataset** (cross-dataset evaluation)

| ส่วนของ test set | จำนวน | label |
|---|---|---|
| Celeb-real | 108 | real |
| YouTube-real | 70 | real |
| Celeb-synthesis | 340 | fake |
| **รวม** (`List_of_testing_videos.txt`) | **518** | |

**Pipeline (ตาม repo):** 8 frame ต่อวิดีโอ แบบเว้นระยะเท่ากัน → RetinaFace → align 5 จุด → crop 224×224 →
main branch → softmax → **เฉลี่ย 8 frame = score ของวิดีโอ** → AUC

**นอกจาก AUC ยังเทียบ score รายวิดีโอกับไฟล์ผลของผู้เขียน** (`checkpoints/Ours/Ours/test/Celeb-DF.csv`) ทีละคลิป
เพื่อดูว่าความต่าง (ถ้ามี) มาจาก preprocessing หรือไม่

**ก่อนรัน:** Accelerator = GPU · Internet = On · ตั้ง notebook เป็น **Private**
ข้อมูล Celeb-DF จะดาวน์โหลดจาก Google Drive ให้อัตโนมัติ (ไม่ต้อง Add Input) — หรือถ้ามี Kaggle Dataset `celebdf_v2_test518`
จาก `prepare_celebdf_test.py` อยู่แล้วก็ Add Input ได้ notebook จะใช้ตัวนั้นก่อน  ·  ใช้เวลาประมาณ 25–45 นาที (รวมดาวน์โหลด ~10 GB)

## 0. ตั้งค่า

In [ ]:
import os, sys, re, time, json, glob, random, shutil, zipfile, hashlib, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
T_START = time.time()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FFD_WORK", "./work")).resolve()
TMP_DIR = Path("/tmp/ffd") if ON_KAGGLE else WORK_DIR / "tmp"
INPUT_DIR = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("FFD_INPUT", str(WORK_DIR / "input")))
REPO_DIR = TMP_DIR / "FFDBackbone"
WEIGHT_DIR = TMP_DIR / "weights"
CROP_DIR = WORK_DIR / "cdf_crops"           # หน้าที่ crop แล้ว (518 วิดีโอ × 8 frame)
OUT_DIR = WORK_DIR / "results_level2_cdf"
for d in [TMP_DIR, WEIGHT_DIR, CROP_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/zhenglab/FFDBackbone.git"
GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1jFIpb4TftJiL82h69c3Roqy8uIzGQP4G"

# --- checkpoint (เหมือนระดับ 1) ---
CKPT_PATH = None
CKPT_NAME_HINTS = ["ours", "final", "ddbf", "dbbf"]
CKPT_EXCLUDE = ["beit-1k-face-55w"]

# --- Celeb-DF ---
CDF_ROOT = None        # None = ค้นหา List_of_testing_videos.txt ใน INPUT_DIR อัตโนมัติ
# ถ้าไม่ได้ Add Input ไว้ จะดาวน์โหลด Celeb-DF-v2.zip (9.95 GB) จาก Google Drive ของเจ้าของ dataset
# ด้วย file id จากลิงก์ที่ได้รับหลังกรอกฟอร์ม แล้วแตกเฉพาะ 518 คลิปของ test set
# (ลิงก์นี้ให้เฉพาะผู้ที่ขอสิทธิ์แล้ว → ตั้ง notebook เป็น Private และอย่าเผยแพร่)
CDF_GDRIVE_ID = "1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj"
CDF_ZIP_BYTES = 9_952_957_051
N_FRAMES = 8           # num_segments ใน config DBBF.yaml
MAX_SHIFT = 10         # ถ้า frame ที่สุ่มได้หาหน้าไม่เจอ ให้ลอง frame ถัดไปได้สูงสุดกี่ frame
PAPER_AUC_CDF = 90.46

# --- preprocessing / inference (ค่าตาม repo) ---
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
DET_THRES = 0.5
BATCH_SIZE = 64
NUM_WORKERS = 2
USE_AMP = True
USE_FUSION = "auto"

# --- analysis ---
UTOM_LAMBDA = 0.02
UTOM_EPS = 1e-7
N_BOOT = 1000
SEED = 42

random.seed(SEED)
print("ON_KAGGLE:", ON_KAGGLE, "| WORK_DIR:", WORK_DIR, "| INPUT_DIR:", INPUT_DIR)

## 1. ติดตั้ง library
ไม่ใช้ `requirements.txt` ของ repo ตรง ๆ เพราะ pin เวอร์ชันเก่า (numpy 1.23, torch 1.13, timm 0.9.2)
ซึ่งจะทำให้ environment ของ Kaggle พัง — ใช้ของที่ Kaggle มีอยู่แล้ว แล้ว patch โค้ด repo แทน (cell ถัดไป)

In [ ]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)

pip_install("gdown")
try:
    import timm  # noqa
except ImportError:
    pip_install("timm")
try:
    import huggingface_hub  # noqa
except ImportError:
    pip_install("huggingface_hub")

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENV = {
    "python": sys.version.split()[0], "torch": torch.__version__, "timm": timm.__version__,
    "numpy": np.__version__, "opencv": cv2.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
print(ENV)

## 2. Clone repo + patch
Patch ที่ทำ (ไม่แตะ logic ของโมเดล):
- `np.float` / `np.int` / `np.bool` → `np.float64` / `np.int64` / `np.bool_` (ถูกลบตั้งแต่ numpy 1.24)
- `np.reshape(..., newshape=...)` → positional (เลิกใช้ใน numpy 2.x)
- เพิ่ม `import torch.utils.model_zoo` ใน RetinaFace loader (torch รุ่นใหม่ไม่ import ให้อัตโนมัติ) และโหลด weight ของ RetinaFace ลง CPU ก่อน

In [ ]:
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
print("repo commit:", REPO_COMMIT)

def patch_repo(repo: Path):
    n_files = 0
    for py in list((repo / "face_process").rglob("*.py")) + list((repo / "models" / "lib" / "BEiT_v2").rglob("*.py")):
        src = py.read_text(encoding="utf-8", errors="ignore")
        new = re.sub(r"np\.float\b", "np.float64", src)
        new = re.sub(r"np\.int\b", "np.int64", new)
        new = re.sub(r"np\.bool\b", "np.bool_", new)
        new = new.replace("np.reshape(b, newshape=(1, 2))", "np.reshape(b, (1, 2))")
        if py.name == "alignment.py":
            if "import torch.utils.model_zoo" not in new:
                new = "import torch.utils.model_zoo\n" + new
            new = new.replace("model_zoo.load_url(url)", "model_zoo.load_url(url, map_location='cpu')")
        if new != src:
            py.write_text(new, encoding="utf-8")
            n_files += 1
    return n_files

print("patched files:", patch_repo(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import importlib.util

def load_module_from_file(name, path):
    """โหลดไฟล์ .py ตรง ๆ — ถ้า import ผ่าน package `models` จะไปเรียก models/__init__.py ซึ่ง import
    backbone ทุกตัว (ต้องใช้ mmpretrain, yacs ฯลฯ ที่ไม่จำเป็นสำหรับงานนี้)"""
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

modeling_finetune = load_module_from_file(                  # BEiT v2 ViT-B/16 ของ repo
    "beitv2_modeling_finetune", REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py")
from face_process.lib.ct.detection import FaceDetector     # RetinaFace (mobilenet0.25) แบบ FTCN
from face_process import face_utils                         # crop_aligned (ArcFace 5-point alignment)

## 3. Checkpoint
ลำดับการหา checkpoint: `CKPT_PATH` → ไฟล์ใน `/kaggle/input` → ดาวน์โหลดจาก Google Drive ด้วย `gdown`

ถ้า gdown ล้มเหลว (Google Drive จำกัดโควตาดาวน์โหลดบ่อย) ให้ดาวน์โหลดไฟล์ checkpoint ของ **Ours** เองจาก
Google Drive → Kaggle: *Add Input → Upload → New Dataset* → ใส่ path ใน `CKPT_PATH` แล้วรันใหม่

In [ ]:
CKPT_EXTS = (".tar", ".pth", ".pt", ".ckpt", ".bin")

def _is_candidate(name: str) -> bool:
    low = name.lower()
    return low.endswith(CKPT_EXTS) and not any(x in low for x in CKPT_EXCLUDE)

def _rank(name: str) -> int:
    low = name.lower()
    return sum(h in low for h in CKPT_NAME_HINTS)

def find_or_download_ckpt():
    if CKPT_PATH and Path(CKPT_PATH).exists():
        return Path(CKPT_PATH)
    # 1) Kaggle input / ไฟล์ที่เคยดาวน์โหลดไว้
    local = [p for root in ["/kaggle/input", str(WEIGHT_DIR)] if Path(root).exists()
             for p in Path(root).rglob("*") if p.is_file() and _is_candidate(p.name)]
    if local:
        local.sort(key=lambda p: (-_rank(str(p)), str(p)))
        print("พบ checkpoint ในเครื่อง:", *local, sep="\n  ")
        return local[0]
    # 2) Google Drive
    import gdown
    print("กำลังอ่านรายชื่อไฟล์ใน Google Drive folder ...")
    import inspect
    kw = dict(url=GDRIVE_FOLDER, skip_download=True, quiet=True)
    if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:   # gdown รุ่นเก่า (<5) จำกัด 50 ไฟล์
        kw["remaining_ok"] = True
    files = gdown.download_folder(**kw)
    entries = []
    for f in files or []:
        fid = getattr(f, "id", None) or (f[0] if isinstance(f, (tuple, list)) else None)
        fpath = getattr(f, "path", None) or (f[1] if isinstance(f, (tuple, list)) else "")
        entries.append((fid, str(fpath)))
    print("ไฟล์ใน Drive:", *[p for _, p in entries], sep="\n  ")
    cands = [(fid, p) for fid, p in entries if _is_candidate(p)]
    if not cands:
        raise RuntimeError("ไม่พบไฟล์ checkpoint ใน Drive — ดาวน์โหลดเองแล้วตั้ง CKPT_PATH")
    cands.sort(key=lambda x: (-_rank(x[1]), x[1]))
    fid, rel = cands[0]
    out = WEIGHT_DIR / rel
    out.parent.mkdir(parents=True, exist_ok=True)
    print(f"ดาวน์โหลด: {rel}")
    gdown.download(id=fid, output=str(out), quiet=False)
    if not out.exists() or out.stat().st_size < 10_000_000:
        raise RuntimeError("ดาวน์โหลดไม่สำเร็จ (อาจติดโควตา Google Drive) — ดาวน์โหลดเองแล้วตั้ง CKPT_PATH")
    return out

def sha256_of(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

ckpt_file = find_or_download_ckpt()
CKPT_SHA = sha256_of(ckpt_file)
print("checkpoint:", ckpt_file, f"({ckpt_file.stat().st_size/1e6:.0f} MB)  sha256:", CKPT_SHA[:16])

### 3.1 สร้างโมเดลและโหลด weight
โครงสร้างเหมือน `DDBF_BEiT_v2` ใน `models/custom.py` (ชื่อ layer ตรงกันเพื่อให้ state dict เข้าได้ตรง ๆ):
- **main branch** = `backbone_model` → LayerNorm → FC — ใช้ตอน inference ตาม paper และ `eval.py`
- **auxiliary branch** = `backbone_model2` (ใช้เฉพาะตอนเทรน) — ถ้า checkpoint มี จะคำนวณผลแบบ uncertainty fusion
  `w = softmax(-10·[u₁, u₂])` เพิ่มเพื่อเปรียบเทียบ
- uncertainty `u = K/S`, `S = Σ(exp(clamp(logit, -10, 10)) + 1)`, K = 2 (Eq. 3–4)

⚠️ `inference.py` ของ repo เก็บเฉพาะ key ที่ขึ้นต้นด้วย `module.` และโหลดแบบ `strict=False` — ถ้า checkpoint ไม่มี prefix นี้
โมเดลจะได้ weight สุ่มโดยไม่มี error ใด ๆ  ใน notebook นี้จึงตรวจว่า **ทุก key ของ main branch ต้องโหลดเข้าได้ครบ**

In [ ]:
BEIT_CONFIG = dict(drop_path_rate=0.1, use_mean_pooling=False, init_values=0.1, qkv_bias=True,
                   use_abs_pos_emb=False, use_rel_pos_bias=True, use_shared_rel_pos_bias=False)

def make_beit():
    m = modeling_finetune.beit_base_patch16_224(**BEIT_CONFIG)
    m.head = nn.Identity()
    return m

def evidential_uncertainty(logits, num_classes=2):
    alpha = torch.exp(torch.clamp(logits, -10, 10)) + 1
    return num_classes / alpha.sum(dim=1, keepdim=True)

class CFBDetector(nn.Module):
    def __init__(self, dual: bool):
        super().__init__()
        self.dual = dual
        self.backbone_model = make_beit()
        if dual:
            self.backbone_model2 = make_beit()
        self.norm_for_cls = nn.LayerNorm(768)
        self.header = nn.Sequential(nn.Linear(768, 2))

    def classify(self, feat):
        return self.header(self.norm_for_cls(feat))

    def forward(self, x):                        # x: [B, 3, 224, 224]
        f1 = self.backbone_model(x)
        out = {"main": self.classify(f1)}
        if self.dual:
            f2 = self.backbone_model2(x)
            l2 = self.classify(f2)
            u = torch.cat([evidential_uncertainty(out["main"].float()), evidential_uncertainty(l2.float())], dim=-1)
            w = torch.softmax(-10 * u, dim=-1).to(f1.dtype)
            out["branch2"] = l2
            out["fused"] = self.classify(w[:, :1] * f1 + w[:, 1:] * f2)
        return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = {k: v for k, v in ck.items() if not isinstance(v, (dict, torch.Tensor))} if isinstance(ck, dict) else {}
    sd = ck
    for key in ("state_dict", "model", "model_state_dict", "net"):
        if isinstance(sd, dict) and isinstance(sd.get(key), dict):
            sd = sd[key]
            break
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items() if torch.is_tensor(v)}
    return sd, meta

state_dict, ckpt_meta = load_state_dict_any(ckpt_file)
HAS_BRANCH2 = any(k.startswith("backbone_model2.") for k in state_dict)
DUAL = HAS_BRANCH2 and USE_FUSION in ("auto", True)
print("checkpoint meta:", {k: ckpt_meta[k] for k in list(ckpt_meta)[:8]})
print("มี auxiliary branch:", HAS_BRANCH2, "| คำนวณ fused output:", DUAL)

model = CFBDetector(dual=DUAL)
expected = set(model.state_dict().keys())
loadable = {k: v for k, v in state_dict.items() if k in expected}
missing = sorted(expected - set(loadable))
unexpected = sorted(set(state_dict) - expected)
shape_bad = [k for k, v in loadable.items() if v.shape != model.state_dict()[k].shape]
print(f"keys: expected={len(expected)} loaded={len(loadable)} missing={len(missing)} unexpected={len(unexpected)}")
if missing or shape_bad:
    raise RuntimeError(f"checkpoint ไม่ตรงกับโมเดล (อาจเลือกไฟล์ผิด เช่น pre-trained backbone)\n"
                       f"missing[:10]={missing[:10]}\nshape mismatch={shape_bad[:10]}")
if unexpected:
    print("unexpected keys (ไม่ใช้):", unexpected[:10], "..." if len(unexpected) > 10 else "")
model.load_state_dict(loadable, strict=True)
model = model.to(DEVICE).eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"โหลดครบ ✓  parameters = {n_params:.1f}M  device = {DEVICE}")
del state_dict, loadable

## 4. Face preprocessing สำหรับวิดีโอ
**การเลือก frame (เลียนแบบ `Image_dataset_test.get_sampled_idx`):** `step = N // 8` แล้วใช้ frame `0, step, 2·step, …` 8 ตัวแรก

**การ crop หน้าแต่ละ frame:** เหมือนระดับ 1 (RetinaFace → align 5 จุด → `crop_face_sbi` phase='test' → 224×224 → RGB)

**ความต่างจากของผู้เขียน (ต้องระบุในรายงาน):**
- ผู้เขียนใช้ pipeline แบบ FTCN: ตรวจหน้า**ทุก frame** แล้ว track หน้าที่ปรากฏนานที่สุด จากนั้นค่อยเลือก 8 frame จาก frame ที่มีหน้า
  (repo ไม่ได้แนบสคริปต์ส่วนนี้มา)
- notebook นี้ตรวจหน้าเฉพาะ 8 frame ที่เลือก, เลือกหน้าแรกด้วยขนาดใหญ่สุด และ frame ถัดไปเลือกหน้าที่ IoU กับ frame ก่อนหน้าสูงสุด (tracking อย่างง่าย)
  ถ้า frame ไหนไม่เจอหน้า จะเลื่อนไปหา frame ถัดไปไม่เกิน `MAX_SHIFT` frame

In [ ]:
detector = FaceDetector(0 if DEVICE.type == "cuda" else -1)

def valid_faces(faces, max_count=10, thres=DET_THRES):
    out = []
    for face in faces[:max_count]:
        box, lm, score = face
        if score < thres:
            continue
        out.append((box.astype(np.float64), lm.astype(np.float64), float(score)))
    return out

def crop_face_test(img, bbox):
    """crop_face_sbi(..., margin=False, crop_by_bbox=True, phase='test') ของ repo"""
    H, W = img.shape[:2]
    (x0, y0), (x1, y1) = bbox
    w, h = x1 - x0, y1 - y0
    w0 = w1 = w / 4 * 0.5
    h0 = h1 = h / 4 * 0.5
    y0n, y1n = max(0, int(y0 - h0)), min(H, int(y1 + h1) + 1)
    x0n, x1n = max(0, int(x0 - w0)), min(W, int(x1 + w1) + 1)
    return img[y0n:y1n, x0n:x1n]

def align_and_crop(img_bgr, face):
    bbox, lm5, _ = face
    aligned, _, bb = face_utils.crop_aligned(img_bgr, lm5, landmarks_68=None, bboxes=bbox, aligned_image_size=IMG_SIZE)
    crop = crop_face_test(aligned, np.array([[bb[0], bb[1]], [bb[2], bb[3]]]))
    if crop.size == 0:
        return None
    crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

def box_iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x1 - x0) * max(0.0, y1 - y0)
    area = lambda r: max(0.0, r[2] - r[0]) * max(0.0, r[3] - r[1])
    return inter / (area(a) + area(b) - inter + 1e-9)

def pick_face(faces, prev_box):
    if prev_box is None:
        return max(faces, key=lambda f: (f[0][2] - f[0][0]) * (f[0][3] - f[0][1]))
    return max(faces, key=lambda f: box_iou(f[0], prev_box))

def sample_indices(n_frames, k=N_FRAMES):
    """เหมือน get_sampled_idx() ของ repo"""
    idx = np.arange(n_frames)
    if n_frames >= k:
        step = n_frames // k
        return idx[::step][:k]
    return np.sort(np.resize(idx, k))     # วิดีโอสั้นกว่า k frame: วนซ้ำ frame เดิม

def read_frames(path, wanted):
    """อ่านวิดีโอแบบ sequential แล้วเก็บเฉพาะ frame ที่ต้องการ (แม่นกว่าการ seek)"""
    wanted = set(int(i) for i in wanted)
    cap = cv2.VideoCapture(str(path))
    frames, i, last = {}, 0, max(wanted) if wanted else -1
    while i <= last:
        ok = cap.grab()
        if not ok:
            break
        if i in wanted:
            ok, fr = cap.retrieve()
            if ok:
                frames[i] = fr
        i += 1
    cap.release()
    return frames, i

def count_frames(path):
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return n, fps, w, h

def process_video(path, out_dir):
    """คืนรายการ dict ต่อ frame ที่ crop ได้ + ข้อมูลวิดีโอ"""
    n, fps, w, h = count_frames(path)
    base = sample_indices(max(n, 1))
    wanted = set()
    for b in base:
        wanted.update(range(int(b), int(b) + MAX_SHIFT + 1))
    frames, n_read = read_frames(path, wanted)
    if n_read < n * 0.9:                     # metadata นับ frame เกินจริง → เลือกใหม่จากจำนวนที่อ่านได้จริง
        base = sample_indices(max(n_read, 1))
        wanted = set()
        for b in base:
            wanted.update(range(int(b), min(int(b) + MAX_SHIFT + 1, n_read)))
        frames, _ = read_frames(path, wanted)
    recs, prev_box, used = [], None, set()
    for k, b in enumerate(base):
        got = None
        for shift in range(MAX_SHIFT + 1):
            fi = int(b) + shift
            if fi in used or fi not in frames:
                continue
            try:
                faces = valid_faces(detector.detect(frames[fi]))
            except Exception:
                faces = []
            if faces:
                face = pick_face(faces, prev_box)
                rgb = align_and_crop(frames[fi], face)
                if rgb is not None:
                    got = (fi, shift, face, len(faces), rgb)
                    break
        if got is None:
            continue
        fi, shift, face, n_faces, rgb = got
        used.add(fi)
        prev_box = face[0]
        dst = out_dir / f"{k}_{fi}.png"
        cv2.imwrite(str(dst), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        recs.append({"slot": k, "frame_idx": fi, "shift": shift, "n_faces": n_faces,
                     "det_score": face[2], "crop_path": str(dst)})
    return recs, {"n_frames_meta": n, "n_frames_read": n_read, "fps": fps, "width": w, "height": h}

def to_tensor(rgb):
    x = (rgb.astype(np.float32) / 255.0 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

@torch.no_grad()
def predict_batch(x):
    x = x.to(DEVICE, non_blocking=True)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
        out = model(x)
    res = {}
    for name, logits in out.items():
        logits = logits.float()
        res[f"p_{name}"] = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        res[f"u_{name}"] = evidential_uncertainty(logits)[:, 0].cpu().numpy()
        res[f"logit_{name}"] = (logits[:, 1] - logits[:, 0]).cpu().numpy()
    return res

## 5. หาข้อมูล Celeb-DF test และอ่านรายชื่อวิดีโอ
ลำดับการหา: Kaggle Dataset ที่ Add Input → zip ใน input → **ดาวน์โหลดจาก Google Drive แล้วแตกเฉพาะ 518 คลิป**

⚠️ ใน `List_of_testing_videos.txt` **1 = real, 0 = fake** (กลับกับโมเดล) → แปลงเป็น `label = 1 − label_official`
เหมือน `dataloader_util.get_data_list()` ของ repo (`FAKE_LABEL - int(line[0])`)

In [ ]:
def extract_test_from_zip(zp):
    """แตกเฉพาะ List_of_testing_videos.txt + 518 คลิปของ test set (ไม่แตกทั้ง 10 GB)"""
    dst = TMP_DIR / "cdf"
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        lst = [n for n in names if n.endswith("List_of_testing_videos.txt")]
        if not lst:
            return None
        prefix = lst[0][: -len("List_of_testing_videos.txt")]          # เช่น "Celeb-DF-v2/"
        z.extract(lst[0], dst)
        wanted = [prefix + l.split()[1] for l in z.read(lst[0]).decode().split("\n") if len(l.split()) == 2]
        have = set(names)
        for n in tqdm([w for w in wanted if w in have], desc="extract test videos"):
            if not (dst / n).exists():
                z.extract(n, dst)
        print(f"แตก {sum(w in have for w in wanted)}/{len(wanted)} คลิปจาก {zp.name}")
    return dst / prefix

def download_cdf_zip():
    import gdown
    zp = TMP_DIR / "Celeb-DF-v2.zip"
    if zp.exists() and zp.stat().st_size == CDF_ZIP_BYTES:
        return zp
    free_gb = shutil.disk_usage(TMP_DIR).free / 1e9
    print(f"ดาวน์โหลด Celeb-DF-v2.zip (9.95 GB) จาก Google Drive · พื้นที่ว่าง {free_gb:.0f} GB")
    gdown.download(id=CDF_GDRIVE_ID, output=str(zp), quiet=False, resume=True)
    if not zp.exists() or zp.stat().st_size != CDF_ZIP_BYTES:
        raise RuntimeError("ดาวน์โหลดไม่ครบ (Google Drive อาจจำกัดโควตา) — ลองรัน cell นี้ใหม่ภายหลัง "
                           "หรือใช้ prepare_celebdf_test.py แล้ว Add Input แทน")
    return zp

def find_cdf_root():
    if CDF_ROOT and (Path(CDF_ROOT) / "List_of_testing_videos.txt").exists():
        return Path(CDF_ROOT)
    # 1) Kaggle Dataset ที่แตกไว้แล้ว
    hits = sorted(INPUT_DIR.rglob("List_of_testing_videos.txt")) if INPUT_DIR.exists() else []
    if hits:
        return hits[0].parent
    # 2) แตกไว้แล้วจากการรันครั้งก่อน
    prev = sorted((TMP_DIR / "cdf").rglob("List_of_testing_videos.txt")) if (TMP_DIR / "cdf").exists() else []
    if prev:
        return prev[0].parent
    # 3) zip ใน input (กรณี Kaggle ไม่แตกให้)
    for zp in (sorted(INPUT_DIR.rglob("*.zip")) if INPUT_DIR.exists() else []):
        root = extract_test_from_zip(zp)
        if root is not None:
            return root
    # 4) ดาวน์โหลดจาก Google Drive
    if CDF_GDRIVE_ID:
        zp = download_cdf_zip()
        root = extract_test_from_zip(zp)
        zp.unlink(missing_ok=True)          # ลบ zip 10 GB หลังแตกเสร็จ เพื่อคืนพื้นที่ดิสก์
        return root
    raise FileNotFoundError("ไม่พบ Celeb-DF test — Add Input หรือกำหนด CDF_GDRIVE_ID")

CDF = find_cdf_root()
rows = []
for line in (CDF / "List_of_testing_videos.txt").read_text().split("\n"):
    parts = line.split()
    if len(parts) == 2:
        rel = parts[1]
        rows.append({"video": rel[:-4], "subset": rel.split("/")[0], "label_official": int(parts[0]),
                     "label": 1 - int(parts[0]), "path": str(CDF / rel)})
videos = pd.DataFrame(rows)
videos["exists"] = videos["path"].map(lambda p: Path(p).exists())
print("CDF root:", CDF)
print(videos.groupby(["subset", "label"]).agg(n=("video", "size"), found=("exists", "sum")))
if not videos["exists"].all():
    print("⚠️ ไม่พบไฟล์:", videos.loc[~videos["exists"], "video"].tolist()[:10])
videos = videos[videos["exists"]].reset_index(drop=True)

## 6. ดึง frame + crop หน้า (ใช้เวลานานที่สุด ~10–20 นาที; รันซ้ำจะใช้ไฟล์เดิม)

In [ ]:
frame_meta_path, video_meta_path = OUT_DIR / "frame_meta.csv", OUT_DIR / "video_meta.csv"
if frame_meta_path.exists() and video_meta_path.exists():
    frame_meta, video_meta = pd.read_csv(frame_meta_path), pd.read_csv(video_meta_path)
    print("ใช้ผล crop เดิม:", len(frame_meta), "frame")
else:
    t0 = time.time()
    f_rows, v_rows = [], []
    for _, r in tqdm(videos.iterrows(), total=len(videos), desc="videos"):
        out_dir = CROP_DIR / r["video"].replace("/", "__")
        out_dir.mkdir(parents=True, exist_ok=True)
        try:
            recs, info = process_video(r["path"], out_dir)
        except Exception as e:
            recs, info = [], {"error": f"{type(e).__name__}: {e}"}
        v_rows.append({"video": r["video"], "n_crops": len(recs), **info})
        for rec in recs:
            f_rows.append({"video": r["video"], **rec})
    frame_meta, video_meta = pd.DataFrame(f_rows), pd.DataFrame(v_rows)
    frame_meta.to_csv(frame_meta_path, index=False)
    video_meta.to_csv(video_meta_path, index=False)
    print(f"crop เสร็จใน {(time.time()-t0)/60:.1f} นาที")

video_meta = videos.merge(video_meta, on="video", how="left")
print("จำนวน frame ที่ crop ได้ต่อวิดีโอ:")
print(video_meta["n_crops"].value_counts().sort_index().to_string())
print("ความละเอียดวิดีโอที่พบบ่อย:", video_meta.groupby(["width", "height"]).size().sort_values().tail(3).to_dict())
print("วิดีโอที่ต้องเลื่อน frame เพื่อหาหน้า:", int((frame_meta["shift"] > 0).groupby(frame_meta["video"]).any().sum()))

## 7. Inference → score ระดับ frame และระดับวิดีโอ (เฉลี่ย softmax ของ 8 frame เหมือน `eval.py`)

In [ ]:
class CropDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return to_tensor(np.asarray(Image.open(self.paths[i]).convert("RGB")))

loader = DataLoader(CropDataset(frame_meta["crop_path"]), batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
t0 = time.time()
outs = [predict_batch(x) for x in tqdm(loader, desc="inference")]
INFER_SEC = time.time() - t0
pred = {k: np.concatenate([o[k] for o in outs]) for k in outs[0]}
frame_scores = pd.concat([frame_meta.reset_index(drop=True), pd.DataFrame(pred)], axis=1)
frame_scores = frame_scores.merge(videos[["video", "subset", "label"]], on="video")
frame_scores.to_csv(OUT_DIR / "frame_scores.csv", index=False)

heads = ["main"] + (["fused", "branch2"] if DUAL else [])
agg = {f"p_{h}": "mean" for h in heads}
agg.update({"u_main": "mean", "slot": "size"})
vid = frame_scores.groupby("video").agg(agg).rename(columns={"slot": "n_frames_used"}).reset_index()
vid = videos[["video", "subset", "label"]].merge(vid, on="video", how="left")
n_noface = int(vid["p_main"].isna().sum())
for h in heads:          # วิดีโอที่หาหน้าไม่เจอเลย: ให้ score 0.5 (ไม่รู้) และรายงานจำนวน
    vid[f"p_{h}"] = vid[f"p_{h}"].fillna(0.5)
vid.to_csv(OUT_DIR / "video_scores.csv", index=False)
print(f"inference {len(frame_scores)} frame ใน {INFER_SEC:.1f} s · วิดีโอที่ไม่มีหน้าเลย: {n_noface}")

## 8. ผลลัพธ์เทียบกับ paper

In [ ]:
def eer_score(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2)

def basic_metrics(y, s, thr=0.5):
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=float)
    pred = (s >= thr).astype(int)
    tnr = float(((pred == 0) & (y == 0)).sum() / max((y == 0).sum(), 1))
    tpr = float(((pred == 1) & (y == 1)).sum() / max((y == 1).sum(), 1))
    return {"n_real": int((y == 0).sum()), "n_fake": int((y == 1).sum()),
            "AUC": roc_auc_score(y, s), "AP": average_precision_score(y, s), "EER": eer_score(y, s),
            "ACC@0.5": float((pred == y).mean()), "BalACC@0.5": (tnr + tpr) / 2,
            "RealAcc(TNR)": tnr, "FakeAcc(TPR)": tpr}

def bootstrap_auc_ci(y, s, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y, s = np.asarray(y), np.asarray(s)
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    aucs = [roc_auc_score(y[idx], s[idx]) for idx in
            (np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]) for _ in range(n_boot))]
    return np.percentile(aucs, [2.5, 97.5])

author = pd.read_csv(REPO_DIR / "checkpoints/Ours/Ours/test/Celeb-DF.csv")
author["video"] = author["video"].str.split("/").str[2:4].str.join("/")
author = author.rename(columns={"predict": "p_author", "label": "label_author"})
vid = vid.merge(author[["video", "p_author", "label_author"]], on="video", how="left")
assert (vid["label"] == vid["label_author"]).all(), "label ไม่ตรงกับไฟล์ของผู้เขียน"

rows = []
for name, col, d in ([("ผู้เขียน (CSV ใน repo)", "p_author", vid)] +
                     [(f"notebook นี้ ({h})", f"p_{h}", vid) for h in heads] +
                     [("notebook นี้ (main, frame-level)", "p_main", frame_scores)]):
    m = basic_metrics(d["label"], d[col])
    lo, hi = bootstrap_auc_ci(d["label"].values, d[col].values)
    rows.append({"source": name, "unit": "frame" if d is frame_scores else "video", **m,
                 "AUC_CI_low": lo, "AUC_CI_high": hi})
results = pd.DataFrame(rows)
results.to_csv(OUT_DIR / "metrics_cdf.csv", index=False)
show = results.copy()
for c in ["AUC", "AP", "EER", "ACC@0.5", "RealAcc(TNR)", "FakeAcc(TPR)", "AUC_CI_low", "AUC_CI_high"]:
    show[c] = (100 * show[c]).round(2)
show["AUC [95% CI]"] = show.apply(lambda r: f"{r['AUC']:.2f} [{r['AUC_CI_low']:.2f}, {r['AUC_CI_high']:.2f}]", axis=1)
print(f"Paper (Table, CFB, Celeb-DF video AUC) = {PAPER_AUC_CDF}\n")
print(show[["source", "unit", "n_real", "n_fake", "AUC [95% CI]", "AP", "EER", "ACC@0.5",
            "RealAcc(TNR)", "FakeAcc(TPR)"]].to_string(index=False))

### 8.1 เทียบ score รายวิดีโอกับของผู้เขียน และแยกตามแหล่งภาพจริง

In [ ]:
from scipy.stats import pearsonr, spearmanr
r_p, _ = pearsonr(vid["p_main"], vid["p_author"])
r_s, _ = spearmanr(vid["p_main"], vid["p_author"])
agree = ((vid["p_main"] >= 0.5) == (vid["p_author"] >= 0.5)).mean()
print(f"Pearson r = {r_p:.3f} · Spearman ρ = {r_s:.3f} · ตัดสินตรงกันที่ τ=0.5: {100*agree:.1f}% ของวิดีโอ")

by_subset = vid.groupby("subset").agg(n=("video", "size"),
                                      mean_p_ours=("p_main", "mean"), mean_p_author=("p_author", "mean"),
                                      acc_ours=("p_main", lambda s: np.nan), acc_author=("p_author", lambda s: np.nan))
for sub in by_subset.index:
    d = vid[vid["subset"] == sub]
    by_subset.loc[sub, "acc_ours"] = ((d["p_main"] >= 0.5).astype(int) == d["label"]).mean()
    by_subset.loc[sub, "acc_author"] = ((d["p_author"] >= 0.5).astype(int) == d["label"]).mean()
print(by_subset.round(3).to_string())
by_subset.to_csv(OUT_DIR / "by_subset.csv")

# AUC แยกแหล่งภาพจริง (Celeb-real vs fake, YouTube-real vs fake)
fake = vid[vid["label"] == 1]
for sub in ["Celeb-real", "YouTube-real"]:
    d = pd.concat([vid[vid["subset"] == sub], fake])
    print(f"AUC ({sub} vs Celeb-synthesis): ours {100*roc_auc_score(d['label'], d['p_main']):.2f} · "
          f"author {100*roc_auc_score(d['label'], d['p_author']):.2f}")

### 8.2 กราฟ

In [ ]:
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK2, GRID = "#52514e", "#e6e5e0"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2})

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
ax = axes[0]
for col, lab, c, ls in [("p_main", "This notebook", C1, "-"), ("p_author", "Authors' CSV", C2, "--")]:
    fpr, tpr, _ = roc_curve(vid["label"], vid[col])
    ax.plot(fpr, tpr, color=c, lw=2, ls=ls, label=f"{lab} (AUC {100*roc_auc_score(vid['label'], vid[col]):.2f})")
ax.plot([0, 1], [0, 1], ls=":", color=INK2, lw=1)
ax.set(xlabel="False positive rate", ylabel="True positive rate", title=f"Celeb-DF v2 video-level ROC (paper {PAPER_AUC_CDF})")
ax.legend(loc="lower right", frameon=False); ax.grid(color=GRID)

ax = axes[1]
bins = np.linspace(0, 1, 26)
for lab, c, name in [(0, C1, "Real"), (1, C2, "Fake")]:
    v = vid.loc[vid["label"] == lab, "p_main"]
    name = f"{name} ({len(v)})"
    ax.hist(v, bins=bins, density=True, alpha=0.25, color=c)
    ax.hist(v, bins=bins, density=True, histtype="step", lw=1.8, color=c, label=name)
ax.axvline(0.5, ls="--", color=INK2, lw=1)
ax.set(xlabel="video score (mean P(fake) over 8 frames)", ylabel="density", title="Score distribution")
ax.legend(frameon=False); ax.grid(axis="y", color=GRID)

ax = axes[2]
for lab, c, name in [(0, C1, "Real"), (1, C2, "Fake")]:
    d = vid[vid["label"] == lab]
    ax.scatter(d["p_author"], d["p_main"], s=14, alpha=0.6, color=c, label=name, edgecolors="none")
ax.plot([0, 1], [0, 1], ls=":", color=INK2, lw=1)
ax.set(xlabel="Authors' video score", ylabel="This notebook's video score",
       title=f"Per-video agreement (Spearman ρ = {r_s:.2f})", xlim=(0, 1), ylim=(0, 1))
ax.legend(frameon=False); ax.grid(color=GRID)
plt.tight_layout()
plt.savefig(OUT_DIR / "cdf_roc_hist_scatter.png", dpi=150)
plt.show()

# วิดีโอที่โมเดลพลาดหนักที่สุด (แถวบน: real ที่ score สูงสุด, แถวล่าง: fake ที่ score ต่ำสุด) — แสดง frame แรกที่ crop ได้
worst_real = vid[vid["label"] == 0].nlargest(6, "p_main")
worst_fake = vid[vid["label"] == 1].nsmallest(6, "p_main")
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for r_i, (dfw, title) in enumerate([(worst_real, "real: highest scores"), (worst_fake, "fake: lowest scores")]):
    for c_i in range(6):
        ax = axes[r_i, c_i]; ax.axis("off")
        if c_i < len(dfw):
            row = dfw.iloc[c_i]
            fr = frame_scores[frame_scores["video"] == row["video"]]
            if len(fr):
                ax.imshow(Image.open(fr.iloc[0]["crop_path"]))
            ax.set_title(f"{row['video'].split('/')[-1]}\np={row['p_main']:.2f}", fontsize=8)
    axes[r_i, 0].text(-0.1, 0.5, title, transform=axes[r_i, 0].transAxes, rotation=90, va="center", ha="right", fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "cdf_worst_errors.png", dpi=120)
plt.show()

## 9. Threshold (Eq. 10–11): หา τ_ot และ τ_ot^UTOM บน Celeb-DF ระดับ face/frame
paper หา threshold จาก **cross-dataset 1 ชุด** แล้วนำไปใช้กับชุดอื่น → บันทึก τ ที่ได้จาก CDF ไว้ใน `thresholds_from_cdf.json`
เพื่อนำไปใช้กับผลระดับ 1 (DeepFakeFace) ได้  ·  ค่า ACC ใน cell นี้เป็นค่า **in-sample** (หา τ และวัดผลบนชุดเดียวกัน) จึงสูงกว่าจริง

In [ ]:
def utom_adjust(p, u, lam=UTOM_LAMBDA, eps=UTOM_EPS):
    return np.tanh(lam * p / (2 * (u + eps)))

def best_threshold(y, s):
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=float)
    order = np.argsort(s)
    s_sorted, y_sorted = s[order], y[order]
    real_below = np.concatenate([[0], np.cumsum(1 - y_sorted)])[:-1]
    fake_at_or_above = y_sorted.sum() - np.concatenate([[0], np.cumsum(y_sorted)])[:-1]
    acc = (real_below + fake_at_or_above) / len(y)
    first = np.r_[True, s_sorted[1:] != s_sorted[:-1]]
    i = np.argmax(np.where(first, acc, -1))
    return float(s_sorted[i]), float(acc[i])

fs = frame_scores
p_adj = utom_adjust(fs["p_main"].values, fs["u_main"].values)
t_ot, acc_ot = best_threshold(fs["label"], fs["p_main"])
t_utom, acc_utom = best_threshold(fs["label"], p_adj)
acc05 = ((fs["p_main"] >= 0.5).astype(int) == fs["label"]).mean()
thr = {"source": "Celeb-DF v2 test (frame level)", "tau_ot": t_ot, "tau_ot_utom": t_utom,
       "lambda": UTOM_LAMBDA, "eps": UTOM_EPS,
       "in_sample_acc": {"tau_0.5": acc05, "tau_ot": acc_ot, "tau_ot_utom": acc_utom}}
json.dump(thr, open(OUT_DIR / "thresholds_from_cdf.json", "w"), indent=2)
print(json.dumps(thr, indent=2))

## 10. บันทึกสรุป

In [ ]:
summary = {
    "config": {"N_FRAMES": N_FRAMES, "MAX_SHIFT": MAX_SHIFT, "DET_THRES": DET_THRES, "USE_AMP": USE_AMP,
               "SEED": SEED, "N_BOOT": N_BOOT},
    "env": ENV, "repo_commit": REPO_COMMIT,
    "checkpoint": {"file": ckpt_file.name, "sha256": CKPT_SHA, "has_branch2": HAS_BRANCH2},
    "n_videos": int(len(vid)), "n_videos_no_face": n_noface, "n_frames": int(len(frame_scores)),
    "paper_auc": PAPER_AUC_CDF,
    "metrics": results.to_dict(orient="records"),
    "agreement_with_author": {"pearson": r_p, "spearman": r_s, "decision_agreement@0.5": agree},
    "thresholds": thr, "inference_seconds": INFER_SEC, "total_minutes": (time.time() - T_START) / 60,
}
json.dump(summary, open(OUT_DIR / "summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
print("ไฟล์ผลลัพธ์ใน", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(" ", p.name)
print(f"เวลารวม {(time.time()-T_START)/60:.1f} นาที")

## หมายเหตุสำหรับเขียนรายงาน
- **Split:** ใช้ official test list 518 วิดีโอ (178 real : 340 fake) ซึ่งตรงกับไฟล์ผลของผู้เขียนทุกคลิป; โมเดลไม่เคยเห็น Celeb-DF ตอนเทรน
- **Label:** ไฟล์ทางการใช้ 1 = real ต้องกลับเป็น 1 = fake ก่อนคำนวณ (repo ทำแบบเดียวกัน)
- **Metric:** video-level AUC = AUC ของค่าเฉลี่ย P(fake) จาก 8 frame; ACC@0.5 ขึ้นกับ threshold และ class imbalance จึงรายงานคู่กับ AUC/EER
- **ถ้า AUC ต่างจาก 90.46:** ดู Spearman ρ กับ score ของผู้เขียน และจำนวนวิดีโอที่ต้องเลื่อน frame / ไม่มีหน้า — ความต่างหลักน่าจะมาจาก
  การเลือก frame และการ track หน้า (ผู้เขียนใช้ FTCN pipeline ที่ไม่ได้เผยแพร่ใน repo)
- **YouTube-real vs Celeb-real:** ภาพจริงสองแหล่งมีคุณภาพต่างกัน ดู `by_subset.csv` ว่าโมเดลพลาดกับแหล่งไหนมากกว่า